In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import py360convert

from PIL import Image

# Detect project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Input directories
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
ANNOTATION_DIR = PROJECT_ROOT / "data" / "annotations"
PANORAMA_DIR = PROJECT_ROOT / "data" / "raw" / "imagery" / "panoramas"

# Output directory
VIEW_DIR = PROCESSED_DIR / "perspective_views"
VIEW_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Panoramas:", PANORAMA_DIR)
print("Output:", VIEW_DIR)

In [ ]:
# Load the frozen 50-location sample
sample_wgs = gpd.read_file(
    PROCESSED_DIR / "panorama_locations.geojson"
)

# The Excel file has no header row
orientations = pd.read_excel(
    ANNOTATION_DIR / "ams_orientation.xlsx",
    header=None,
    names=["location_id", "orientation"]
)

# Clean Excel values
orientations["location_id"] = (
    orientations["location_id"].astype(str).str.strip()
)

orientations["orientation"] = (
    orientations["orientation"].astype(str).str.strip().str.upper()
)

# Validate before generating anything
expected_ids = set(sample_wgs["location_id"])
provided_ids = set(orientations["location_id"])

assert len(orientations) == 50, "Expected 50 orientation records."
assert orientations["location_id"].is_unique, "Duplicate location IDs."
assert expected_ids == provided_ids, "Missing or unexpected location IDs."
assert orientations["orientation"].isin(["A", "B"]).all(), "Invalid orientation."

# Join classifications to location metadata
dataset = sample_wgs.merge(
    orientations,
    on="location_id",
    how="left",
    validate="one_to_one"
)

print("Locations:", len(dataset))
print("\nOrientation distribution:")
print(dataset["orientation"].value_counts())

display(
    dataset[
        ["location_id", "orientation", "timestamp"]
    ].head(10)
)

In [ ]:
ANGLE_PAIRS = {
    "A": (0, 180),
    "B": (90, 270)
}

VIEW_SIZE = 768
FOV = 90

image_records = []

for _, row in dataset.sort_values("location_id").iterrows():

    location_id = row["location_id"]
    orientation = row["orientation"]

    pano_path = PANORAMA_DIR / f"{location_id}.jpg"

    if not pano_path.exists():
        print(f"Missing panorama: {location_id}")
        continue

    pano = np.asarray(
        Image.open(pano_path).convert("RGB")
    )

    for view_label, angle in zip(
        ["A", "B"],
        ANGLE_PAIRS[orientation]
    ):

        view = py360convert.e2p(
            pano,
            fov_deg=FOV,
            u_deg=angle,
            v_deg=0,
            out_hw=(VIEW_SIZE, VIEW_SIZE)
        )

        # Crop the bottom 10% to reduce camera/vehicle visibility
        view = view[:int(VIEW_SIZE * 0.90), :, :]

        filename = f"{location_id}_{view_label}.jpg"
        output_path = VIEW_DIR / filename

        Image.fromarray(view).save(
            output_path,
            quality=95
        )

        image_records.append({
            "image_id": f"{location_id}_{view_label}",
            "location_id": location_id,
            "orientation_pair": orientation,
            "view_angle": angle,
            "filename": filename,
            "pano_id": row["pano_id"],
            "timestamp": row["timestamp"]
        })

    print(f"{location_id} → completed")

# Save image metadata
image_df = pd.DataFrame(image_records)

image_df.to_csv(
    PROCESSED_DIR / "perspective_views_metadata.csv",
    index=False
)

print("\nGenerated images:", len(image_df))
print("Unique locations:", image_df["location_id"].nunique())